# 01 - Análise Exploratória (EDA)

## 1.0 Contexto de Problema

Baseado em uma estrutura de dados de uma campanha de marketing de uma instituição bancária com o objetivo de fazer conversões do produto _bank term deposit_ por intermédio de contatos telefônicos, elaborar uma plataforma de experimentação adaptativa para a oferta com uma solução end-to-end de Machine Learning Engineering.

A Base de dados escolhida foi a: bank-term-deposit-subscription (dharmik34)
Link: https://www.kaggle.com/datasets/dharmik34/bank-term-deposit-subscription/data

Neste Notebook será elaborada a analise exploratória dos dados. 

## 2.0 Setup e Carga

In [1]:
# Importando bibliotecas necessárias

import pandas as pd
import numpy as np

In [2]:
# Carregando data set
df_bank = pd.read_csv('../data/raw/bank-full.csv', sep=';')

## 3.0 Visão geral dos dados

In [3]:
# Informaçoes da tabela bank-full.csv
df_bank.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


### 3.1 Resumo

- Linhas: 45.211
- Colunas: 17
- Uso de Memória: 5.9 MB
- Estrutura de dados coerentes com relação a tipo
- Não apresenta valores nulos.

## 4.0 Análise da Base

### 4.1 Análise do target

Análise da taxa de conversão da campanha bem como a taxa de conversão em função do **braço**: _contact_.

In [4]:
# Identificação das classes da coluna  "y" (objetivo)
df_bank['y'].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [5]:
# Taxa de conversão geral
conv_rate = df_bank['y'].value_counts(normalize=True)
conv_rate_yes = conv_rate['yes']

print(f"Taxa de conversão geral: {conv_rate_yes: .2%}")

Taxa de conversão geral:  11.70%


In [6]:
# Identificação das classes da coluna "contact" (braço) 

df_bank['contact'].value_counts()

contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

In [7]:
# Taxa de conversão por braço
conv_rate_arm = pd.crosstab(df_bank['contact'], df_bank['y'], normalize = 'index') * 100
print (conv_rate_arm)


y                 no        yes
contact                        
cellular   85.081100  14.918900
telephone  86.579491  13.420509
unknown    95.929339   4.070661


É possível identificar uma quantidade de tipos de contato desconhecidos relevante em função do tamanho da base, com uma taxa de conversão pequena mas não desprezível.

São 13020 cadastros de contato desconhecidos com uma conversão de 4,07% para essa classificação. Portanto é necessário explorar mais a fim de entender se _unknown_ é um canal de contato desconhecido ou se são falta de registro do tipo de contato.

### 4.2 - Análise da coluna _unknown_

In [8]:
# Análise da distribuição do tipo de contato por mês

pd.crosstab(df_bank['contact'], df_bank['month'])

month,apr,aug,dec,feb,jan,jul,jun,mar,may,nov,oct,sep
contact,,,,,,,,,,,,
cellular,2727,5950,174,2362,1265,5767,729,417,5331,3540,557,466
telephone,199,246,37,274,129,852,80,53,460,379,131,66
unknown,6,51,3,13,9,276,4532,7,7975,51,50,47


In [9]:
#Análise da distribuição do tipo de contato em resultado de campanhas anteriores. 

pd.crosstab(df_bank['contact'], df_bank['poutcome'])

poutcome,failure,other,success,unknown
contact,,,,
cellular,4529,1637,1385,21734
telephone,341,176,114,2275
unknown,31,27,12,12950


É possível verificar uma incidência concentrada de _contact_ = _unknown_ nos meses de maio e junho que somam 12.507 das 13.020, aproximadamente 96% do total o que não se vê nos outros meses.

O cruzamento com _poutcome_ reforça a leitura: 12.950 das 13.020 linhas com _contact_ = _unknown_ são as mesmas que apresentam _poutcome_ = _unknown_, ou 
seja, clientes sem participação em campanha anterior. Trata-se de um lote de primeiro contato.

Unindo essa informação com a descrição da base de dados no Kaggle que declara que os dados são disponibilizados a partir de maio de 2008, podemos inferir que essa quantidade de dados _unknown_ pode vir de uma fase em que o processo de coleta dos dados ainda não era maduro e não se fazia registro dessa informação. 

Assim, podemos assumir que _contact_ classificado como _unknown_ é ausência de registro e não um canal de atendimento. A consequência para a modelagem é direta: esse grupo apresenta taxa de conversão de 4,07%, contra 14,92% do cellular e 13,42% do telephone. Mantê-lo como braço faria o modelo adaptativo aprender a evitar um canal inexistente, atribuindo à alavanca um efeito que decorre do perfil dos clientes e do período. As 13.020 linhas são portanto excluídas da base de treinamento do modelo adaptativo, restando dois braços: cellular (29.285 registros) e telephone (2.906).

In [10]:
# Eliminando contatos desconhecidos

df_bank_clean = df_bank[(df_bank['contact'] != 'unknown')].copy()

### 4.3 Análise das features

Análise de inconsistências em cada coluna do banco de dados.

In [11]:
# Contagem de unknown nas colunas job, education e poutcome

for col in ['job', 'education', 'poutcome']:
    n = (df_bank_clean[col] == 'unknown').sum()
    print(f"{col}: {n} ({n/len(df_bank_clean):.1%})")

job: 177 (0.5%)
education: 1180 (3.7%)
poutcome: 24009 (74.6%)


A quantidade de _unknown_ para as colunas job, education é pequena em comparação do total então permanecem na elaboração do modelo.
poutcome tem valor alto e a hipótese é de que são clientes que nunca sofreram nenhum contato. Para confirmar podemos comparar a quantidade de ocorrência de poutcome = 'unknown', pdays = -1 e previous = 0.

In [12]:
p_unk = df_bank_clean['poutcome'] == 'unknown'
pdays_neg = df_bank_clean['pdays'] == -1
prev_zero = df_bank_clean['previous'] == 0

print(p_unk.sum(), pdays_neg.sum(), prev_zero.sum())
print("as três coincidem:", (p_unk & pdays_neg & prev_zero).sum())

24009 24004 24004
as três coincidem: 24004


Confirmada a hipótese, sabemos que os dados são reais mas precisam ser ajustados: a coluna `pdays` carrega duas informações. O valor -1 indica que o cliente nunca foi contatado; os demais valores indicam há quantos dias ocorreu o último contato. Assim, é necessário dividir em duas colunas.

In [13]:
# Separando informação da coluna pdays em 2 e removendo a mesma

df_bank_clean['contatado_antes'] = (df_bank_clean['pdays'] != -1).astype(int)
df_bank_clean['dias_ultimo_contato'] = df_bank_clean['pdays'].replace(-1, 0)

df_bank_clean = df_bank_clean.drop('pdays', axis=1)
df_bank_clean.info()



<class 'pandas.DataFrame'>
Index: 32191 entries, 12657 to 45210
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   age                  32191 non-null  int64
 1   job                  32191 non-null  str  
 2   marital              32191 non-null  str  
 3   education            32191 non-null  str  
 4   default              32191 non-null  str  
 5   balance              32191 non-null  int64
 6   housing              32191 non-null  str  
 7   loan                 32191 non-null  str  
 8   contact              32191 non-null  str  
 9   day                  32191 non-null  int64
 10  month                32191 non-null  str  
 11  duration             32191 non-null  int64
 12  campaign             32191 non-null  int64
 13  previous             32191 non-null  int64
 14  poutcome             32191 non-null  str  
 15  y                    32191 non-null  str  
 16  contatado_antes      32191 non-nul

In [14]:
# Separando informação da coluna poutcome em 2 e removendo a mesma

df_bank_clean['poutcome'] = df_bank_clean['poutcome'].replace(
    'unknown', 'sem_campanha_anterior'
)

In [15]:
# Análise da feature 'balance'

df_bank_clean['balance'].describe()

count     32191.000000
mean       1434.557982
std        3191.590608
min       -8019.000000
25%          82.000000
50%         478.000000
75%        1515.000000
max      102127.000000
Name: balance, dtype: float64

Da análise de balance é possível verificar que temos assimetria da distribuição dos dados e valores negativos o que dificulta a implementação de um escalador linear. Assim serão criadas faixas de valor por quantil de 20%.

**Conclusão da análise das features**

- A classificação _unknown_ em `job` (177 registros, 0,5%) e `education` (1.180, 3,7%) tem volume baixo e é mantida como categoria própria.

- `poutcome` = _unknown_ representa 24.009 registros (74,6%), mas não é ausência de registro: indica cliente sem campanha anterior. A verificação cruzada com `pdays` = -1 e `previous` = 0 aponta para o mesmo conjunto de linhas, com apenas 5 divergências. A categoria é renomeada para `sem_campanha_anterior`.

- `pdays` carrega duas informações em um único campo: se houve contato anterior (-1 para não) e há quantos dias ocorreu. O valor -1 seria interpretado pelo modelo como quantidade negativa de dias. A coluna é 
  desmembrada em `contatado_antes` (binária) e `dias_ultimo_contato` (0 quando não houve contato).

- `balance` é fortemente assimétrica: mediana de 478 contra máximo de 102.127, com desvio padrão (3.191) superior à média (1.434). Valores negativos correspondem a saldo devedor e são legítimos. A escala direta 
  comprimiria a maior parte da base, portanto a coluna será discretizada em faixas por quantil na camada de features.

### 4.4 Comparação dos braços por segmento

Para verificar se algum perfil de cliente responde melhor ao telefone, compara-se a taxa de conversão dos dois braços dentro de cada categoria das features disponíveis. São consideradas apenas células com ao menos 100 registros em cada braço.



In [16]:
# Target numérico e faixas para análise por segmento

df_bank_clean['convertido'] = (df_bank_clean['y'] == 'yes').astype(int)

bins   = [17, 30, 40, 50, 60, 100]
labels = ['18-30', '31-40', '41-50', '51-60', '60+']
df_bank_clean['age_band'] = pd.cut(df_bank_clean['age'], bins=bins, labels=labels)

df_bank_clean['balance_band'] = pd.qcut(
    df_bank_clean['balance'], q=5,
    labels=['muito_baixo', 'baixo', 'medio', 'alto', 'muito_alto']
)

In [17]:
def busca_interacao(df, features, min_n=100):
    """Compara a conversão dos dois braços dentro de cada categoria."""
    resultados = []

    for feat in features:
        g = (df.groupby([feat, 'contact'], observed=True)['convertido']
               .agg(['size', 'mean'])
               .unstack('contact'))

        if ('size', 'telephone') not in g.columns:
            continue

        res = pd.DataFrame({
            'feature':   feat,
            'categoria': g.index.astype(str),
            'n_cel':     g[('size', 'cellular')],
            'n_tel':     g[('size', 'telephone')],
            'conv_cel':  g[('mean', 'cellular')]  * 100,
            'conv_tel':  g[('mean', 'telephone')] * 100,
        }).reset_index(drop=True)

        resultados.append(res)

    out = pd.concat(resultados, ignore_index=True)
    out['diff'] = out['conv_tel'] - out['conv_cel']
    out = out[(out['n_cel'] >= min_n) & (out['n_tel'] >= min_n)]

    return out.sort_values('diff', ascending=False).round(2)


features = ['job', 'marital', 'education', 'housing', 'loan',
            'poutcome', 'age_band', 'balance_band']

interacoes = busca_interacao(df_bank_clean, features)

# Apenas os extremos, para leitura
pd.concat([interacoes.head(5), interacoes.tail(5)])

,feature,categoria,n_cel,n_tel,conv_cel,conv_tel,diff
5,job,retired,1370,405,26.72,31.85,5.14
12,marital,divorced,3326,271,14.61,18.45,3.84
18,education,unknown,990,190,19.39,20.53,1.13
15,education,primary,3568,722,11.63,12.60,0.97
36,balance_band,muito_alto,5706,731,20.47,21.34,0.87
9,job,technician,5381,321,13.58,8.41,-5.17
14,marital,single,8791,620,18.80,13.06,-5.74
10,job,unemployed,891,100,19.42,13.00,-6.42
27,age_band,18-30,4800,299,20.69,14.05,-6.64
31,age_band,60+,799,372,45.06,38.17,-6.88


In [18]:
# Verificação do único segmento com vantagem relevante para telephone

retired = df_bank_clean[df_bank_clean['job'] == 'retired']

rate = pd.crosstab([retired['age_band'], retired['contact']],
                   retired['y'], normalize='index') * 100
vol  = pd.crosstab([retired['age_band'], retired['contact']], retired['y'])

pd.concat([vol, rate.round(2)], axis=1)

y                    no  yes      no    yes
age_band contact                           
18-30    cellular     3    0  100.00   0.00
31-40    cellular    16    0  100.00   0.00
         telephone    1    0  100.00   0.00
41-50    cellular    44    2   95.65   4.35
         telephone    7    2   77.78  22.22
51-60    cellular   623   99   86.29  13.71
         telephone   97   17   85.09  14.91
60+      cellular   318  265   54.55  45.45
         telephone  171  110   60.85  39.15

In [19]:
# Colunas criadas apenas para esta análise
df_bank_clean = df_bank_clean.drop(columns=['convertido', 'age_band', 'balance_band'])

Das 33 células com volume suficiente, apenas 6 apresentam vantagem para o telefone, e somente uma supera 4 pontos percentuais. O celular vence nas 27 restantes, com diferenças de até 6,9 pontos e em segmentos de grande volume. no celular é, portanto, superior de forma praticamente uniforme.

O único segmento com vantagem relevante para o telefone é `job` = _retired_ (31,85% contra 26,72%). Ao controlar pela faixa etária, entretanto, o efeito se inverte: na faixa 60+ o celular converte 45,45% contra 39,15% do telefone. A vantagem agregada decorre da composição da carteira — 69,4% dos aposentados 
contatados por telefone estão na faixa 60+, contra 42,6% dos contatados por celular. Como essa faixa converte a taxas muito superiores às demais, o telefone se beneficia de um mix mais favorável, e não da eficácia do canal. Trata-se de um paradoxo de Simpson.

**Consequências para a modelagem**

1. Não existe segmento em que o telefone supere o celular de forma sustentada. Um modelo contextual não teria variação de decisão a aprender.

2. O canal de contato não foi atribuído aleatoriamente nesta base. A comparação direta entre braços mistura o efeito do canal com o perfil dos clientes selecionados para cada um. Essa é a limitação central do trabalho e justamente o problema que uma plataforma de experimentação adaptativa resolve, ao aleatorizar a decisão e gerar dados adequados para aprendizado.

3. O baseline adotado será a alocação uniforme (50/50), equivalente a um teste A/B tradicional, e não o melhor braço histórico. A instituição não conhece de antemão qual canal é superior; o ganho da política adaptativa está na velocidade com que ela identifica o melhor braço e reduz o tráfego desperdiçado.

### 4.5 Remoção de vazamentos

Nessa base temos a coluna `duration` que indica o tempo de duração das chamadas. Essa informação só é sabida após a realização da chamada. 
Usar essa informação em um modelo  que deve prever qual o melhor canal de contato seria um vazamento temporal e portanto deve ser removida.

In [20]:
# Remoção da Coluna 'duration'

df_bank_clean = df_bank_clean.drop(columns=['duration'])

df_bank_clean.info()

<class 'pandas.DataFrame'>
Index: 32191 entries, 12657 to 45210
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   age                  32191 non-null  int64
 1   job                  32191 non-null  str  
 2   marital              32191 non-null  str  
 3   education            32191 non-null  str  
 4   default              32191 non-null  str  
 5   balance              32191 non-null  int64
 6   housing              32191 non-null  str  
 7   loan                 32191 non-null  str  
 8   contact              32191 non-null  str  
 9   day                  32191 non-null  int64
 10  month                32191 non-null  str  
 11  campaign             32191 non-null  int64
 12  previous             32191 non-null  int64
 13  poutcome             32191 non-null  str  
 14  y                    32191 non-null  str  
 15  contatado_antes      32191 non-null  int64
 16  dias_ultimo_contato  32191 non-nul

In [21]:
df_bank_clean.describe()

,age,balance,day,campaign,previous,contatado_antes,dias_ultimo_contato
count,32191.000000,32191.000000,32191.000000,32191.000000,32191.000000,32191.000000,32191.000000
mean,41.072163,1434.557982,15.986922,2.754279,0.806157,0.254326,56.676400
std,11.057405,3191.590608,8.261942,2.980085,2.678185,0.435488,112.246503
min,18.000000,-8019.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,32.000000,82.000000,9.000000,1.000000,0.000000,0.000000,0.000000
50%,39.000000,478.000000,16.000000,2.000000,0.000000,0.000000,0.000000
75%,49.000000,1515.000000,21.000000,3.000000,1.000000,1.000000,13.500000
max,95.000000,102127.000000,31.000000,50.000000,275.000000,1.000000,871.000000


In [22]:
df_bank_clean['poutcome'].value_counts()

poutcome
sem_campanha_anterior    24009
failure                   4870
other                     1813
success                   1499
Name: count, dtype: int64

## 5.0 Exportação da camada trusted

Os dados tratados são gravados em formato Parquet na camada _trusted_. As colunas categóricas recebem o tipo `category` e o alvo `y` é convertido para binário (1 = conversão).

O formato Parquet preserva o schema junto com os dados, evitando a reinferência de tipos que ocorreria em CSV, e reduz o volume em disco pelo armazenamento colunar com dicionário para categóricas.

Esta camada contém o dado limpo e íntegro, sem transformações específicas de
modelo. Codificação de categóricas, escala e criação de faixas pertencem à camada _refined_, construída no notebook seguinte.

In [23]:
# Tipagem
categoricas = ['job', 'marital', 'education', 'default', 'housing',
               'loan', 'contact', 'month', 'poutcome']

for col in categoricas:
    df_bank_clean[col] = df_bank_clean[col].astype('category')

# Target como binário
df_bank_clean['y'] = (df_bank_clean['y'] == 'yes').astype(int)

In [24]:
# Exportação

df_bank_clean.to_parquet('../data/trusted/bank_clean.parquet', index=False)

In [25]:
teste = pd.read_parquet('../data/trusted/bank_clean.parquet')
teste.info()

<class 'pandas.DataFrame'>
RangeIndex: 32191 entries, 0 to 32190
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   age                  32191 non-null  int64   
 1   job                  32191 non-null  category
 2   marital              32191 non-null  category
 3   education            32191 non-null  category
 4   default              32191 non-null  category
 5   balance              32191 non-null  int64   
 6   housing              32191 non-null  category
 7   loan                 32191 non-null  category
 8   contact              32191 non-null  category
 9   day                  32191 non-null  int64   
 10  month                32191 non-null  category
 11  campaign             32191 non-null  int64   
 12  previous             32191 non-null  int64   
 13  poutcome             32191 non-null  category
 14  y                    32191 non-null  int64   
 15  contatado_antes      32191 non